🛠️ Task 1 — Data Loading, Merging & Deep Exploration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# 1. Load Primary Superstore Sales Dataset
df_store = pd.read_csv('train.csv')

# 2. Parse Datetime Objects
# Note: Superstore dataset dates are typically formatted as DD/MM/YYYY or MM/DD/YYYY. 
# Setting dayfirst=True handles the standard day-month-year parsing securely.
df_store['Order Date'] = pd.to_datetime(df_store['Order Date'], dayfirst=True, errors='coerce')
df_store['Ship Date'] = pd.to_datetime(df_store['Ship Date'], dayfirst=True, errors='coerce')

# Drop missing targets if any fail parsing
df_store = df_store.dropna(subset=['Order Date', 'Sales'])

# 3. Extract Chronological Time Features
df_store['Year'] = df_store['Order Date'].dt.year
df_store['Month'] = df_store['Order Date'].dt.month
df_store['Week_Number'] = df_store['Order Date'].dt.isocalendar().week
df_store['Day_of_Week'] = df_store['Order Date'].dt.day_name()
df_store['Quarter'] = df_store['Order Date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df_store['Season'] = df_store['Month'].apply(get_season)

# Calculate Shipping Lead Time Velocity
df_store['Shipping_Days'] = (df_store['Ship Date'] - df_store['Order Date']).dt.days

# 4. Integrity Verification (Missing values & duplicates)
print("=== DATA INTEGRITY PROFILE ===")
print(f"Total Rows: {df_store.shape[0]} | Columns: {df_store.shape[1]}")
print(f"Missing Values across rows:\n{df_store.isnull().sum()[df_store.isnull().sum() > 0]}")
print(f"Duplicate Rows Count: {df_store.duplicated().sum()}\n")

# 5. Load & Aggregate Supplementary Dataset (Video Game Sales)
df_games = pd.read_csv('vgsales.csv')
df_games = df_games.dropna(subset=['Year'])
df_games['Year'] = df_games['Year'].astype(int)
# Sum up annual market demand
df_games_annual = df_games.groupby('Year')['Global_Sales'].sum().reset_index()
df_games_annual.rename(columns={'Global_Sales': 'External_Gaming_Demand'}, inplace=True)

# 6. Aggregate Primary Store Data into Weekly & Monthly Baselines
df_monthly_raw = df_store.set_index('Order Date').resample('MS')['Sales'].sum().to_frame().reset_index()
df_monthly_raw['Year'] = df_monthly_raw['Order Date'].dt.year
df_monthly_raw['Month'] = df_monthly_raw['Order Date'].dt.month

# Complete the Multi-Source Merge onto the Monthly Timeline
df_merged_monthly = pd.merge(df_monthly_raw, df_games_annual, on='Year', how='left')
df_weekly = df_store.set_index('Order Date').resample('W')['Sales'].sum().to_frame()

print("=== TRANSFORMATION SUCCESSFUL ===")
print(f"Monthly Merged Baseline Profile Dimensions: {df_merged_monthly.shape}")
print(f"Weekly Baseline Profile Dimensions: {df_weekly.shape}\n")


# === 7. ANSWERING STRATEGIC BUSINESS QUESTIONS WITH DATA DATA-BACKED MATRICES ===
print("=== BUSINESS DATA RESPONSES ===")

# Q1: Revenue Generation Category
revenue_cat = df_store.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print("\n1. Total Revenue Generation Across Core Categories:")
display(revenue_cat.apply(lambda x: f"${x:,.2f}"))

# Q2: Growth Consistency by Region over 4 years
regional_growth = df_store.groupby(['Region', 'Year'])['Sales'].sum().unstack(level=0)
print("\n2. Regional Financial Progression Matrix (Yearly Revenue):")
display(regional_growth.applymap(lambda x: f"${x:,.2f}"))

# Q3: Shipping Lead Variance by Region
ship_variance = df_store.groupby('Region')['Shipping_Days'].agg(['mean', 'median', 'std'])
print("\n3. Shipping Delay Thresholds (Order to Ship Window in Days):")
display(ship_variance.round(2))

# Q4: Seasonality Patterns (Spike Months Across Years)
seasonal_spikes = df_store.groupby(['Year', 'Month'])['Sales'].sum().unstack(level=0)
print("\n4. Cross-Year Monthly Seasonal Heatmap Blueprint:")
display(seasonal_spikes.applymap(lambda x: f"${x:,.2f}"))

=== TASK 1: MULTI-SOURCE DATA INTELLIGENCE COMPILED ===
Merged Dataset Shape: 48 months across mapped years.



,Year,Month,Sales,Date,External_Gaming_Market_Demand
0,2015,1,14205.707,2015-01-01,264.44
1,2015,2,4519.892,2015-02-01,264.44
2,2015,3,55205.797,2015-03-01,264.44
3,2015,4,27906.855,2015-04-01,264.44
4,2015,5,23644.303,2015-05-01,264.44


--- Analytical Explorations ---
1. Highest Total Revenue Generating Category (Superstore):
Category
Technology         827,455.87
Furniture          728,658.58
Office Supplies    705,422.33
Name: Sales, dtype: str
